# Transient Navier-Stokes flow around a cylinder with a user weak-form class

A nonlinear user weak form is written by subclassing
`fedoo.core.weakform.WeakFormBase`{.interpreted-text role="class"}:
`update` stores the current velocity, pressure and velocity gradient at
the Gauss points, `set_start` keeps the velocity of the last converged
increment, and `get_weak_equation` returns the weak form linearized at
the current state (tangent operator plus residual) for the
Newton-Raphson solver of `fedoo.problem.NonLinear`{.interpreted-text
role="class"}.

For an incompressible Newtonian fluid, with
$\boldsymbol{\sigma} = -p\,\mathbf{I} + 2\mu\,\mathbf{D}(\mathbf{v})$,

$$\int_\Omega \rho \left( \frac{\partial \mathbf{v}}{\partial t}
+ (\nabla \mathbf{v})\,\mathbf{v} \right) \cdot \delta\mathbf{v}
+ \boldsymbol{\sigma} : \nabla \delta\mathbf{v} \, d\Omega = 0, \qquad
\int_\Omega \delta p \, \nabla \cdot \mathbf{v} \, d\Omega = 0,$$

with a backward Euler time derivative. The mixed problem uses the stable
`p1+/p1` element (bubble-enriched velocity, linear pressure).

The Reynolds number based on the cylinder diameter and on the final
inflow velocity is $Re = \rho U (2R) / \mu = 2000$, and 100 nonlinear
time increments are computed. The inflow velocity is ramped and only
reaches $U = 1$ at the last increment. This example is illustrative and
the flow is not resolved: the mesh Reynolds number $\rho U h / \mu$ is
about 500 in the wake, far above what the unstabilized Galerkin
formulation used here can resolve, and a converged solution at this
Reynolds number would be unsteady.


In [ ]:
import fedoo as fd
import numpy as np
from fedoo.core.weakform import WeakFormBase

# Nonlinear weak form linearized at the current state


In [ ]:
class NavierStokes(WeakFormBase):
    """
    Navier-Stokes weak formulation for incompressible Newtonian fluids.

    This implementation uses a mixed velocity/pressure formulation.
    It supports transient and non-linear convection terms.

    Parameters
    ----------
    rho: float or array
        Density of the fluid.
    mu: float or array
        Dynamic viscosity of the fluid.
    name: str, optional
        Name of the weak form.
    space: ModelingSpace, optional
        Modeling space.
    """

    def __init__(self, rho, mu, name="", space=None):
        super().__init__(name, space)
        self.rho = rho
        self.mu = mu

        # Velocity variables
        self.space.new_variable("Vx")
        self.space.new_variable("Vy")
        if self.space.ndim == 3:
            self.space.new_variable("Vz")
            self.space.new_vector("V", ["Vx", "Vy", "Vz"])
        else:
            self.space.new_vector("V", ["Vx", "Vy"])

        # Pressure variable
        self.space.new_variable("P")

        # Internal operators for updating sv
        self._op_v = [
            self.space.variable(name) for name in ["Vx", "Vy", "Vz"][: self.space.ndim]
        ]
        self._op_p = self.space.variable("P")

        self._grad_ops = []
        for i in range(self.space.ndim):
            var_name = ["Vx", "Vy", "Vz"][i]
            for j in range(self.space.ndim):
                crd_name = ["X", "Y", "Z"][j]
                self._grad_ops.append(self.space.derivative(var_name, crd_name))

    def initialize(self, assembly, pb):
        """Initialize state variables."""
        assembly.sv["V"] = np.zeros((self.space.ndim, assembly.n_gauss_points))
        assembly.sv["P"] = np.zeros(assembly.n_gauss_points)
        assembly.sv["gradV"] = np.zeros((self.space.ndim**2, assembly.n_gauss_points))
        self._v_start = np.zeros((self.space.ndim, assembly.n_gauss_points))

    def set_start(self, assembly, pb):
        """Called at the beginning of a new time step."""
        self._v_start = assembly.sv["V"].copy()

    def update(self, assembly, pb):
        """Update current values from solution."""
        sol = pb.get_dof_solution()
        if np.isscalar(sol) and sol == 0:
            return

        v_results = []
        for op in self._op_v:
            v_results.append(assembly.get_gp_results(op, sol))
        assembly.sv["V"] = np.array(v_results)
        assembly.sv["P"] = assembly.get_gp_results(self._op_p, sol)

        grads = []
        for op in self._grad_ops:
            grads.append(assembly.get_gp_results(op, sol))
        assembly.sv["gradV"] = np.array(grads)

    def get_weak_equation(self, assembly, pb):
        """Build the weak equation including tangent and residual."""
        ndim = self.space.ndim

        # Increments (operators)
        V = [self.space.variable(["Vx", "Vy", "Vz"][i]) for i in range(ndim)]
        P = self.space.variable("P")

        # grad_V_inc[i][j] = dVi / dXj
        grad_V_inc = [[None for _ in range(ndim)] for _ in range(ndim)]
        for i in range(ndim):
            for j in range(ndim):
                grad_V_inc[i][j] = self.space.derivative(
                    ["Vx", "Vy", "Vz"][i], ["X", "Y", "Z"][j]
                )

        # Current state
        V_curr = assembly.sv["V"]
        P_curr = assembly.sv["P"]
        gradV_curr = assembly.sv["gradV"].reshape(ndim, ndim, -1)

        # Time step
        dt = pb.dtime

        DiffOp = 0

        # 1. Momentum Equations (Conservation of linear momentum)
        for i in range(ndim):
            # Viscous part: 2*mu*D_ij * delta_V_i,j = mu*(V_i,j + V_j,i) * delta_V_i,j
            for j in range(ndim):
                # Residual part
                res_visc = self.mu * (gradV_curr[i, j] + gradV_curr[j, i])
                # Tangent part
                tan_visc = self.mu * (grad_V_inc[i][j] + grad_V_inc[j][i])

                DiffOp += grad_V_inc[i][j].virtual * (res_visc + tan_visc)

            # Pressure part in momentum equation: -P * div(delta_v)
            # = -P * delta_Vi,i
            DiffOp += grad_V_inc[i][i].virtual * (-P_curr - P)

            # Convection part: rho * (V . grad) V
            # Residual: rho * sum_j ( Vj_curr * gradVi,j_curr )
            res_conv = self.rho * sum(V_curr[j] * gradV_curr[i, j] for j in range(ndim))
            # Tangent: rho * sum_j ( dVj * gradVi,j_curr + Vj_curr * dgradVi,j )
            tan_conv = self.rho * sum(
                V[j] * gradV_curr[i, j] + V_curr[j] * grad_V_inc[i][j]
                for j in range(ndim)
            )

            # Time derivative: rho * dV/dt
            if dt != 0:
                res_time = (self.rho / dt) * (V_curr[i] - self._v_start[i])
                tan_time = (self.rho / dt) * V[i]
            else:
                res_time = 0
                tan_time = 0

            DiffOp += V[i].virtual * (res_conv + tan_conv + res_time + tan_time)

        # 2. Continuity Equation (Conservation of mass: div(V) = 0)
        # Residual: div(V_curr)
        res_cont = sum(gradV_curr[i, i] for i in range(ndim))
        # Tangent: div(dV)
        tan_cont = sum(grad_V_inc[i][i] for i in range(ndim))

        DiffOp += P.virtual * (res_cont + tan_cont)

        # Axisymmetric case
        if self.space._dimension == "2Daxi":
            rr = assembly.sv["_R_gausspoints"]
            # Add extra term for divergence in cylindrical coordinates: Vr/r
            # Vr is Vx (radial)
            res_cont_axi = V_curr[0] / rr
            tan_cont_axi = V[0] / rr
            DiffOp += P.virtual * (res_cont_axi + tan_cont_axi)

            # Also extra viscous term: sigma_theta = -P + 2*mu*Vr/r
            res_visc_axi = 2 * self.mu * V_curr[0] / rr
            tan_visc_axi = 2 * self.mu * V[0] / rr
            # Virtual work: sigma_theta * delta_eps_theta = sigma_theta * delta_Vr / r
            DiffOp += V[0].virtual * (
                1 / rr * (res_visc_axi + tan_visc_axi - P_curr - P)
            )

            DiffOp = DiffOp * ((2 * np.pi) * rr)

        return DiffOp

# Modeling space, fluid properties and weak form


In [ ]:
space = fd.ModelingSpace("2D")

rho = 10.0  # density
mu = 0.1  # dynamic viscosity (low value to see some convection effects)

# NavierStokes weak form defined above
wf = NavierStokes(rho, mu)

# Mesh: flow around a cylinder (hole in a plate) with a wake region


In [ ]:
L1 = 100
R = 10
mesh1 = fd.mesh.hole_plate_mesh(
    length=L1, height=L1, radius=R, elm_type="tri3", sym=False
)
# Add a wake region
mesh2 = fd.mesh.rectangle_mesh(
    nx=40,
    ny=20,
    x_min=L1 / 2,
    x_max=L1 / 2 + 200,
    y_min=-L1 / 2,
    y_max=L1 / 2,
    elm_type="tri3",
)

mesh = fd.Mesh.stack(mesh1, mesh2)
mesh.merge_nodes(mesh.find_coincident_nodes())

# Stable mixed element (P1+ / P1)

P1+ elements (linear triangles with bubble functions) for velocity, P1
elements for pressure.


In [ ]:
internal_nodes = mesh.add_internal_nodes(1)

elm_type = "p1+/p1"
new_elm = fd.lib_elements.CombinedElement(elm_type, "tri3")
new_elm.set_variable_interpolation("Vx", "tri3bubble")
new_elm.set_variable_interpolation("Vy", "tri3bubble")
new_elm.set_variable_interpolation("P", "tri3")

# Assembly, problem and boundary conditions

The Dirichlet values are ramped linearly from 0 over the simulated time
(default time function). The outflow (right edge) is traction free,
which also fixes the pressure level.

The two left corners belong both to the inlet and to a wall: they are
removed from the inlet nodes so that they keep the no-slip value (two
Dirichlet conditions with different values on the same dof would emit a
`BoundaryConditionWarning`, the last one added prevailing).


In [ ]:
assembly = fd.Assembly.create(wf, mesh, elm_type=elm_type, name="assembling")
pb = fd.problem.NonLinear("assembling")
res = pb.add_output("results/navier_stokes", assembly, ["P", "V"])

left = mesh.find_nodes("X", mesh.bounding_box.xmin)
right = mesh.find_nodes("X", mesh.bounding_box.xmax)
bottom = mesh.find_nodes("Y", mesh.bounding_box.ymin)
top = mesh.find_nodes("Y", mesh.bounding_box.ymax)
hole = mesh.find_nodes("Point", np.array([0, 0]), R + 1e-3)
inlet = np.setdiff1d(left, np.hstack((bottom, top)))  # left edge without corners

# Inlet velocity (horizontal inflow, no vertical velocity)
pb.bc.add("Dirichlet", inlet, "Vx", 1.0)
pb.bc.add("Dirichlet", inlet, "Vy", 0.0)

# No-slip on top and bottom (walls)
pb.bc.add("Dirichlet", bottom, "V", 0.0)
pb.bc.add("Dirichlet", top, "V", 0.0)

# No-slip on the cylinder
pb.bc.add("Dirichlet", hole, "V", 0.0)

# The pressure dof of the bubble nodes is not used by the P1 pressure interpolation:
# it is set to zero to remove it from the system.
pb.bc.add("Dirichlet", internal_nodes, "P", 0.0)

# Solve


In [ ]:
pb.nlsolve(dt=1, tmax=100, update_dt=True)

# Pressure field and streamlines at the last increment

Streamlines are not part of the fedoo plot function: they are added with
PyVista on the plotted mesh (`res.meshplot`), with a zero out-of-plane
velocity component.


In [ ]:
pl = res.plot("P", show=False, title="Pressure")

res.meshplot["velocity"] = np.c_[res["V"].T, np.zeros(mesh.n_nodes)]
line_streamlines = res.meshplot.streamlines(
    "velocity",
    pointa=(mesh.bounding_box.xmin + 1, mesh.bounding_box.ymin + 1, 0),
    pointb=(mesh.bounding_box.xmin + 1, mesh.bounding_box.ymax - 1, 0),
    n_points=25,
    max_length=500.0,
)

pl.add_mesh(line_streamlines.tube(radius=0.5), color="white")
pl.show()